# PAXG vs. Gold Perpetual: 1-Minute Technical-Analysis Basis Arbitrage

Same pair as `paxg_gold_perp_trend_arb.ipynb` (PAXG spot vs. OKX's XAU-USDT-SWAP gold perpetual), but
tested with a genuinely different method: **1-minute intraday data**, with the trend and mean-reversion
signals built from classic technical-analysis constructs instead of daily rolling-regression slopes.

**Trend lines, made tradeable without lookahead bias**: a literal "connect the swing highs/lows"
trendline (drawn in Section 3 for visual intuition, same as the daily notebook) can't be used directly
as a backtest signal — confirming a swing pivot requires seeing bars *after* it, which is lookahead
bias if used at the moment it happens. The tradeable trend signal here is a **rolling channel
breakout** (price vs. the rolling N-minute high/low) — fully causal, and one of the oldest, most
literal trend-line-based systematic techniques there is (this is the Donchian channel breakout
method). Price breaking above the rolling high = uptrend breakout; below the rolling low = downtrend
breakout; inside the channel = no active trend.

**Price deviation**: the same basis z-score construct as the daily notebook (deviation of the basis
from its own rolling average), just with a much shorter, intraday-appropriate lookback window.

**Combining them**: block new mean-reversion entries on the basis while the underlying is in an
active channel breakout (either direction) — same filtering logic as the daily notebook, just with
the trend definition swapped from "rolling regression slope" to "channel breakout."

**Data**: OKX's public API, 1-minute candles, paginated back to build several days of history. Real
constraint worth stating up front: OKX's 1-minute pagination is real (confirmed — each paginated call
reliably advances exactly 100 minutes further back), but building a multi-day window costs dozens of
sequential requests per instrument; this notebook targets roughly the last several days as a
practical balance between real intraday sample size and fetch time.

> ⚠️ Research only, not investment advice. No transaction costs, funding payments, or slippage
> modeled. At 1-minute resolution, trade count will be far higher than the daily version — Section 8's
> cost check matters even more here.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection (1-Minute, Paginated)

`TARGET_MINUTES` controls how far back to fetch — each paginated call advances ~100 minutes, so e.g.
7200 minutes (5 days) costs roughly 72 requests per instrument. Increase if you want a longer window
(and are prepared to wait longer / risk hitting a rate limit); this notebook keeps it modest.


In [ ]:
OKX_BASE = "https://www.okx.com/api/v5/market"
TARGET_MINUTES = 7200  # ~5 days


def fetch_okx_minutes(inst_id, target_minutes=TARGET_MINUTES, bar="1m"):
    all_rows = []

    r = requests.get(f"{OKX_BASE}/candles", params={"instId": inst_id, "bar": bar, "limit": 100}, timeout=20)
    r.raise_for_status()
    data = r.json()["data"]
    all_rows.extend(data)
    after = min(int(row[0]) for row in data)

    while len(all_rows) < target_minutes:
        r = requests.get(
            f"{OKX_BASE}/history-candles",
            params={"instId": inst_id, "bar": bar, "limit": 100, "after": after},
            timeout=20,
        )
        r.raise_for_status()
        data = r.json()["data"]
        if not data:
            break
        all_rows.extend(data)
        after = min(int(row[0]) for row in data)
        time.sleep(0.2)

    cols = ["ts", "open", "high", "low", "close", "vol", "volCcy", "volCcyQuote", "confirm"]
    df = pd.DataFrame(all_rows, columns=cols)
    df = df[df["confirm"] == "1"]
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="ms")
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "vol"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "vol"]]


print("Fetching PAXG-USDT 1-minute history...")
paxg = fetch_okx_minutes("PAXG-USDT")
print("Fetching XAU-USDT-SWAP 1-minute history...")
perp = fetch_okx_minutes("XAU-USDT-SWAP")

print(f"\nPAXG-USDT:     {paxg.index.min()} to {paxg.index.max()}, {len(paxg)} bars")
print(f"XAU-USDT-SWAP: {perp.index.min()} to {perp.index.max()}, {len(perp)} bars")


## 2. Basis Construction


In [ ]:
df = pd.DataFrame({
    "paxg_close": paxg["close"],
    "perp_close": perp["close"],
}).dropna()

df["basis_pct"] = df["perp_close"] / df["paxg_close"] - 1

print(f"Overlapping sample: {df.index.min()} to {df.index.max()}, {len(df)} minutes")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["paxg_close"], label="PAXG (spot)", color="tab:blue")
axes[0].plot(df.index, df["perp_close"], label="XAU-USDT-SWAP (perp)", color="tab:orange", alpha=0.8)
axes[0].set_title("PAXG Spot vs. Gold Perpetual — 1-Minute")
axes[0].legend()

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green", linewidth=0.6)
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("Basis: (Perp / PAXG - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("p < 0.05 suggests the basis is stationary (mean-reverting) over this sample.")


## 3. Trend Lines: Geometric (Visual) and Channel Breakout (Tradeable)

Left: literal trend lines connecting recent swing highs/lows, for visual intuition — this is what
"trend lines" means informally, but it can't be used as a live signal without lookahead bias (you
need future bars to confirm a swing point was real). Right: the rolling-channel breakout signal
actually used in the backtest below — fully causal, and a standard trend-following construct in its
own right (Donchian channel breakouts).


In [ ]:
def find_pivots(price, window=20):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=4):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).total_seconds().values / 60.0
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


recent_window = df.iloc[-2000:]  # last ~33 hours, for a readable chart
swing_highs, swing_lows = find_pivots(recent_window["paxg_close"], window=20)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(recent_window.index, recent_window["paxg_close"], color="tab:blue", label="PAXG close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=20, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=20, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, recent_window.index[-1], freq="min")
        minutes_elapsed = (x_line - x0).total_seconds() / 60.0
        y_line = intercept + slope * minutes_elapsed
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("PAXG (Recent Window): Swing-High / Swing-Low Trend Lines (Visual Only)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
CHANNEL_WINDOW = 30  # minutes

df["channel_high"] = df["paxg_close"].rolling(CHANNEL_WINDOW).max().shift(1)  # shift(1): don't use current bar's own extreme
df["channel_low"] = df["paxg_close"].rolling(CHANNEL_WINDOW).min().shift(1)

df["breakout_up"] = df["paxg_close"] > df["channel_high"]
df["breakout_down"] = df["paxg_close"] < df["channel_low"]
df["in_breakout"] = df["breakout_up"] | df["breakout_down"]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(recent_window.index, recent_window["paxg_close"], color="tab:blue", linewidth=0.8, label="PAXG close")
ax.plot(recent_window.index, df.loc[recent_window.index, "channel_high"], color="red", linewidth=0.8,
        linestyle="--", label=f"{CHANNEL_WINDOW}min rolling high")
ax.plot(recent_window.index, df.loc[recent_window.index, "channel_low"], color="green", linewidth=0.8,
        linestyle="--", label=f"{CHANNEL_WINDOW}min rolling low")
breakout_pts = recent_window.index[df.loc[recent_window.index, "in_breakout"]]
ax.scatter(breakout_pts, recent_window.loc[breakout_pts, "paxg_close"], color="black", s=8, zorder=3,
           label="breakout bar")
ax.set_title(f"PAXG (Recent Window): {CHANNEL_WINDOW}-Minute Channel Breakout Signal (Tradeable)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Breakout-up: {df['breakout_up'].sum()} bars, breakout-down: {df['breakout_down'].sum()} bars, "
      f"total breakout: {df['in_breakout'].mean()*100:.1f}% of the sample")


## 4. Does a Channel Breakout Predict Basis Deviation?


In [ ]:
def zscore(s, window=60):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

Z_WINDOW = 60  # minutes
df["basis_z"] = zscore(df["basis_pct"], window=Z_WINDOW)

breakout_basis = df.loc[df["in_breakout"], "basis_z"].dropna()
calm_basis = df.loc[~df["in_breakout"], "basis_z"].dropna()

print(f"Mean |basis_z| during channel breakouts: {breakout_basis.abs().mean():.3f} (n={len(breakout_basis)})")
print(f"Mean |basis_z| outside breakouts:         {calm_basis.abs().mean():.3f} (n={len(calm_basis)})")

fig, ax = plt.subplots()
ax.hist(calm_basis, bins=40, alpha=0.5, density=True, label="outside breakout")
ax.hist(breakout_basis, bins=40, alpha=0.5, density=True, label="during breakout")
ax.set_title("Basis Z-Score Distribution: Breakout vs. Calm Periods")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Mean-Reversion Arbitrage Signal (on the Basis)


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)
print(f"Pure MR: {(position_pure_mr.diff().abs() > 0).sum()} trades over {len(df)} minutes")


## 6. Combining Both Ideas: Channel-Breakout-Filtered Arbitrage


In [ ]:
position_breakout_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=df["in_breakout"])

print(f"Minutes blocked by breakout filter: {df['in_breakout'].sum()} / {len(df)} "
      f"({df['in_breakout'].mean()*100:.1f}%)")
print(f"Breakout-filtered: {(position_breakout_filtered.diff().abs() > 0).sum()} trades")


In [ ]:
MINUTES_PER_YEAR = 365 * 24 * 60


def sharpe(returns, periods_per_year=MINUTES_PER_YEAR):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "breakout_filtered_mean_reversion": position_breakout_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Breakout-Filtered Basis Arbitrage (1-Minute)")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(MINUTES_PER_YEAR),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "breakout_filtered_mean_reversion": (position_breakout_filtered.diff().abs() > 0).sum(),
    }),
})
summary


> ⚠️ **Read this before trusting the Sharpe numbers above**: annualizing a backtest this short by
> multiplying by `sqrt(minutes_per_year)` amplifies noise enormously — a few days of unusually calm,
> tightly-mean-reverting basis behavior can produce a Sharpe ratio in the double or triple digits that
> has no real relationship to what a full year would actually look like. Treat `annualized_sharpe`
> here as a diagnostic for comparing the two strategies *against each other on this same sample*, not
> as a credible estimate of real annual risk-adjusted return. `total_return` (the actual, un-annualized
> return over the ~5 days fetched) is the more honest number to anchor on.


## 7. Market-Neutrality Check


In [ ]:
paxg_ret = np.log(df["paxg_close"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_gold": [strategies["pure_mean_reversion"].corr(paxg_ret)],
    "breakout_filtered_vs_gold": [strategies["breakout_filtered_mean_reversion"].corr(paxg_ret)],
}, index=["correlation_with_paxg_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?

This is the section that matters most at 1-minute resolution — trade counts here will be far higher
than the daily version, and each one still costs the same ~0.05%-per-fill assumption.


In [ ]:
TAKER_FEE_PER_FILL = 0.0005
FILLS_PER_ROUND_TRIP = 4

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways

This run's actual findings (re-run periodically — a 5-day window will look different each time):

- **The channel-breakout filter barely mattered here, and in an unexpected direction**: Section 4
  showed basis deviation was actually *smaller* during breakout minutes (0.70) than calm minutes
  (0.93) — the opposite of "trending price destabilizes the basis." Breakouts were also rare (9.5% of
  the sample), so the filtered strategy ended up nearly identical to pure mean-reversion (352 vs. 354
  trades, near-identical Sharpe and return). On this window, the technical-analysis trend filter
  added essentially nothing.
- **The headline number is trade frequency, not the trend filter**: 1-minute z-score mean-reversion
  generated **~354 trades in 5 days** — compare that to the daily notebook's 25-76 trades over
  *years*. That difference is the entire story of why Section 8's fee drag is so much worse here
  (≈70% of notional over 5 days) than in the daily version.
- **Sample size is real but short**: ~5 days of 1-minute bars is enough to see intraday structure,
  nowhere near enough for a confident conclusion about a repeatable edge — and per the warning after
  Section 6, don't trust the annualized Sharpe number at face value.
- **Compare Section 4's breakout-vs-calm basis dispersion to the daily notebook's trend-correlation
  finding**: same underlying question (does trending price destabilize the basis), answered with a
  different, more literally technical-analysis operationalization of trend. They don't obviously
  agree here — worth checking whether that holds up over a longer window.
- **Data collection cost is real**: fetching this took over a hundred paginated OKX requests across
  both instruments. If extending to a much longer window, consider caching fetched minutes to disk
  (same pattern as the `flightDataResearch` notebook's `data_cache/`) so re-runs don't refetch
  everything.
- **Robustness**: as always, validate out-of-sample and account for realistic costs before treating
  any of this as an actual edge — and at this trade frequency, costs are the dominant factor, not the
  signal quality.
